# Online Newspaper Database

A walkthrough of the CMPE351 database: create the schema, load the sample data, and run a few queries.

Run it from the repository folder (Jupyter), or in Google Colab, where the first cell fetches the code.

In [ ]:
# In Google Colab, fetch the project files first. In Jupyter from the repo folder this does nothing.
import os
if not os.path.exists("newspaper.py"):
    !git clone -q https://github.com/cemakcck/online-newspaper-database.git
    %cd online-newspaper-database

In [ ]:
import pandas as pd
import newspaper

conn = newspaper.connect(":memory:")
newspaper.create_schema(conn)
newspaper.seed(conn)

def show(sql, params=()):
    return pd.read_sql_query(sql, conn, params=params)

show("SELECT name AS table_name FROM sqlite_master WHERE type = 'table' ORDER BY name")

## Articles and their authors

In [ ]:
pd.DataFrame(newspaper.articles_with_authors(conn), columns=["Article", "Authors"])

## Who read what (JOIN across USER, ReadBy and ARTICLE)

In [ ]:
pd.DataFrame(newspaper.articles_read_by_users(conn), columns=["User", "Article", "Category"])

## Create, update and delete

In [ ]:
new_id = newspaper.add_article(conn, "Deprem Hazırlığı Rehberi", "Yaşam", "2024-04-01")
newspaper.update_article_category(conn, new_id, "Güvenlik")
show("SELECT * FROM ARTICLE WHERE Article_ID = ?", (new_id,))

In [ ]:
# Deleting a user also removes their reads and subscription (ON DELETE CASCADE).
newspaper.delete_user(conn, 1)
show("SELECT * FROM ReadBy WHERE User_ID = 1")

For the interactive menu, run `python newspaper.py` in a terminal.